In [66]:
import numpy as np
import pandas as pd

train_data = pd.read_csv('train_data.csv')
test_data = pd.read_csv('test_data.csv')

In [67]:
x = train_data[['open','high','low','volume']].to_numpy(dtype=float)
y = train_data[['close']].to_numpy(dtype=float)


In [68]:
features = ['open', 'high', 'low', 'volume']

# Preserve every row from the test dataset
x_test = test_data[features].copy()

# Use medians calculated from the training dataset
medians = train_data[features].median()

# Fill missing feature values instead of dropping rows
x_test = x_test.fillna(medians)

# Convert to NumPy
x_test = x_test.to_numpy(dtype=float)

print("Test dataset:", len(test_data))
print("x_test:", len(x_test))

Test dataset: 10558
x_test: 10558


In [69]:
x_mean = x_train.mean(axis=0)
x_std = x_train.std(axis=0)
x_std[x_std == 0] = 1.0

x_train = (x_train - x_mean) / x_std
x_test = (x_test - x_mean) / x_std

In [70]:
# Generate predictions for every test row
y_pred_scaled = x_test @ weights + bias

# Convert predictions back to the original Close-price scale
y_pred = y_pred_scaled * y_std + y_mean

print("Test dataset:", len(test_data))
print("X_test:", len(x_test))
print("Predictions:", len(y_pred))


Test dataset: 10558
X_test: 10558
Predictions: 10558


In [71]:
n_samples, n_features = x_train.shape
weights = np.zeros(n_features)
bias = 0.0

learning_rate = 0.01
epochs = 2000

In [72]:
for epoch in range(epochs):
    y_pred_train = x_train @ weights + bias
    error = y_pred_train - y_train_scaled

    dw = (2 / n_samples) * (x_train.T @ error)
    db = (2 / n_samples) * np.sum(error)

    weights -= learning_rate * dw
    bias -= learning_rate * db

    if (epoch + 1) % 200 == 0:
        loss = np.mean(error ** 2)
        print(f"Epoch {epoch + 1}, MSE: {loss:.6f}")


Epoch 200, MSE: 0.172175
Epoch 400, MSE: 0.172078
Epoch 600, MSE: 0.172073
Epoch 800, MSE: 0.172073
Epoch 1000, MSE: 0.172073
Epoch 1200, MSE: 0.172073
Epoch 1400, MSE: 0.172073
Epoch 1600, MSE: 0.172073
Epoch 1800, MSE: 0.172073
Epoch 2000, MSE: 0.172073


In [73]:
y_pred = (x_test @ weights + bias) * y_std + y_mean

In [74]:
if 'Close' in test_data.columns:
    y_test = test_data['Close'].to_numpy(dtype=float)
    valid = np.isfinite(y_test)

    if valid.any():
        print("\nModel Evaluation")
        print("MAE:", mean_absolute_error(y_test[valid], y_pred[valid]))
        print("MSE:", mean_squared_error(y_test[valid], y_pred[valid]))

        if valid.sum() >= 2:
            print("R2 Score:", r2_score(y_test[valid], y_pred[valid]))


In [75]:
results = pd.DataFrame({'Predicted_Close': y_pred})

if 'Close' in test_data.columns:
    results.insert(0, 'Actual_Close', test_data['Close'].to_numpy())

results.to_csv("stock_predictions.csv", index=False)

print("\nPredictions:")
print(results.head(10))
print("\nSaved to stock_predictions.csv")


Predictions:
   Predicted_Close
0    -1.162061e+06
1    -1.482247e+07
2    -5.213699e+06
3    -2.701971e+06
4    -6.628292e+05
5    -1.361009e+07
6    -4.879574e+06
7    -5.225691e+05
8    -4.460972e+06
9    -2.486766e+06

Saved to stock_predictions.csv


In [76]:
print("Test dataset:", len(test_data))
print("x_test:", len(x_test))
print("y_pred:", len(y_pred))

print("Missing Close values:", test_data["close"].isna().sum())

Test dataset: 10558
x_test: 10558
y_pred: 10558
Missing Close values: 0


In [77]:
import numpy as np

y_test = test_data['close'].to_numpy(dtype=float)

valid = np.isfinite(y_test) & np.isfinite(y_pred)

actual = y_test[valid]
predicted = y_pred[valid]

errors = actual - predicted

mae = np.mean(np.abs(errors))
mse = np.mean(errors ** 2)
rmse = np.sqrt(mse)

ss_res = np.sum(errors ** 2)
ss_tot = np.sum((actual - np.mean(actual)) ** 2)

r2 = 1 - ss_res / ss_tot if ss_tot != 0 else float('nan')

print("Valid test rows:", len(actual))
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)



Valid test rows: 10558
MAE: 6402392.636743776
MSE: 200702564368351.38
RMSE: 14166953.249317631
R2 Score: -108650776098.86513


In [ ]:
import numpy as np

print("X train range:",
      np.nanmin(x_train), np.nanmax(x_train))

print("Y train range:",
      np.nanmin(y_train), np.nanmax(y_train))

print("Y test range:",
      np.nanmin(y_test), np.nanmax(y_test))

print("Predictions range:",
      np.nanmin(y_pred), np.nanmax(y_pred))

print("NaN in y_test:", np.isnan(y_test).sum())
print("NaN in predictions:", np.isnan(y_pred).sum())


X train range: -3.3991248928690316 52.37464628571587
Y train range: 0.0 1578.130005
Y test range: 5.92749975 464.309998
Predictions range: -267733321.35595012 -65383.90186638451
NaN in y_test: 0
NaN in predictions: 0


: 